# 06_gurobi_realistic_model

This notebook builds and solves the realistic childcare expansion model using the Part 2 inputs exported by Notebook 05.

Main tasks:
1. load realistic-model parameter files
2. build the realistic MILP with candidate-site constraints
3. solve the model with Gurobi
4. export solution tables and validation checks for reporting


In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 200)

In [3]:
# Project paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

REALISTIC_DIR = PROJECT_ROOT / "data" / "processed" / "realistic"
RESULTS_DIR = PROJECT_ROOT / "results" / "realistic"
SOLUTIONS_DIR = RESULTS_DIR / "solutions"
LOG_DIR = RESULTS_DIR / "logs"

SOLUTIONS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("REALISTIC_DIR:", REALISTIC_DIR)
print("SOLUTIONS_DIR:", SOLUTIONS_DIR)
print("LOG_DIR:", LOG_DIR)

PROJECT_ROOT: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project
REALISTIC_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/data/processed/realistic
SOLUTIONS_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/realistic/solutions
LOG_DIR: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/realistic/logs


In [4]:
# Check required input files
required_files = [
    REALISTIC_DIR / "zipcode_demand_supply_realistic.csv",
    REALISTIC_DIR / "facility_expansion_params_realistic.csv",
    REALISTIC_DIR / "candidate_build_options_realistic.csv",
    REALISTIC_DIR / "candidate_candidate_conflicts.csv",
    REALISTIC_DIR / "candidate_existing_conflicts.csv",
]

missing_files = [str(f) for f in required_files if not f.exists()]

print("Required input files check:")
for path in required_files:
    print(path.name, "exists ->", path.exists())

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

Required input files check:
zipcode_demand_supply_realistic.csv exists -> True
facility_expansion_params_realistic.csv exists -> True
candidate_build_options_realistic.csv exists -> True
candidate_candidate_conflicts.csv exists -> True
candidate_existing_conflicts.csv exists -> True


In [5]:
# Load realistic input tables
zip_df = pd.read_csv(REALISTIC_DIR / "zipcode_demand_supply_realistic.csv")
fac_df = pd.read_csv(REALISTIC_DIR / "facility_expansion_params_realistic.csv")
cand_df = pd.read_csv(REALISTIC_DIR / "candidate_build_options_realistic.csv")
cc_conflict_df = pd.read_csv(REALISTIC_DIR / "candidate_candidate_conflicts.csv")
ce_conflict_df = pd.read_csv(REALISTIC_DIR / "candidate_existing_conflicts.csv")

In [6]:
# Quick preview
print("zipcode_demand_supply_realistic:", zip_df.shape)
display(zip_df.head())

print("facility_expansion_params_realistic:", fac_df.shape)
display(fac_df.head())

print("candidate_build_options_realistic:", cand_df.shape)
display(cand_df.head())

zipcode_demand_supply_realistic: (311, 23)


,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609.0,24.0,9.0,58.0,472.0,1,1,1
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724.0,198.0,56.0,0.0,1230.0,0,1,1
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995.0,0.0,7.0,0.0,960.0,0,1,1
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263.0,0.0,2.0,0.0,289.0,0,1,1
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39.0,0.0,1.0,363.0,323.0,1,1,1


facility_expansion_params_realistic: (7712, 18)


,facility_id,facility_name,program_type,zipcode,latitude,longitude,n_f,b_f,cap1,cap2,cap3,cap10_cum,cap15_cum,cap20_cum,U_f_realistic,coef1,coef2,coef3
0,51349,"Valerio, Gladys",FDC,10474,40.818399,-73.888816,6,6,0,0,1,0,0,1,1,3533.333333,3733.333333,4333.333333
1,73544,YMCA OF GREATER NY,SACC,10017,40.753216,-73.970794,60,0,6,3,3,6,9,12,12,533.333333,733.333333,1333.333333
2,108407,"Almond Tree Group Family Day Care, LLC",GFDC,11225,NaN,NaN,16,12,1,1,1,1,2,3,3,1450.000000,1650.000000,2250.000000
3,111953,"Williams, Petal",GFDC,11226,NaN,NaN,16,12,1,1,1,1,2,3,3,1450.000000,1650.000000,2250.000000
4,126425,"Hernandez, Lidia",GFDC,10465,40.841228,-73.823572,12,12,1,0,1,1,1,2,2,1866.666667,2066.666667,2666.666667


candidate_build_options_realistic: (85566, 10)


,candidate_id,zipcode,latitude,longitude,geo_missing_flag,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost,blocked_by_existing
0,1,10001,40.741893,-74.000140,False,small,100,50,65000,0
1,1,10001,40.741893,-74.000140,False,medium,200,100,95000,0
2,1,10001,40.741893,-74.000140,False,large,400,200,115000,0
3,2,10001,40.752007,-74.005436,False,small,100,50,65000,0
4,2,10001,40.752007,-74.005436,False,medium,200,100,95000,0


## Model preparation

The preparation stage converts the exported realistic parameter files into the typed sets and dictionaries needed to build the MILP cleanly in Gurobi.


In [7]:
# Final type cleanup before modeling
def standardize_zipcode(series):
    return (
        series.astype(str)
              .str.strip()
              .str.replace(r"\.0$", "", regex=True)
              .str.zfill(5)
    )

zip_df["zipcode"] = standardize_zipcode(zip_df["zipcode"])
fac_df["zipcode"] = standardize_zipcode(fac_df["zipcode"])
cand_df["zipcode"] = standardize_zipcode(cand_df["zipcode"])

fac_df["facility_id"] = fac_df["facility_id"].astype(str)
cand_df["candidate_id"] = pd.to_numeric(cand_df["candidate_id"], errors="coerce").astype(int)
cand_df["size"] = cand_df["size"].astype(str)

if not cc_conflict_df.empty:
    cc_conflict_df["zipcode"] = standardize_zipcode(cc_conflict_df["zipcode"])
    cc_conflict_df["candidate_id_1"] = pd.to_numeric(cc_conflict_df["candidate_id_1"], errors="coerce").astype(int)
    cc_conflict_df["candidate_id_2"] = pd.to_numeric(cc_conflict_df["candidate_id_2"], errors="coerce").astype(int)

required_zip_cols = ["zipcode", "req_total", "req_under5", "existing_total", "existing_under5"]
required_fac_cols = [
    "facility_id", "zipcode", "n_f", "b_f",
    "cap1", "cap2", "cap3", "coef1", "coef2", "coef3"
]
required_cand_cols = [
    "candidate_id", "zipcode", "size", "new_total_capacity",
    "new_under5_capacity_max", "fixed_build_cost"
]

missing_zip_cols = [c for c in required_zip_cols if c not in zip_df.columns]
missing_fac_cols = [c for c in required_fac_cols if c not in fac_df.columns]
missing_cand_cols = [c for c in required_cand_cols if c not in cand_df.columns]

if missing_zip_cols:
    raise KeyError(f"Missing columns in zipcode_demand_supply_realistic.csv: {missing_zip_cols}")
if missing_fac_cols:
    raise KeyError(f"Missing columns in facility_expansion_params_realistic.csv: {missing_fac_cols}")
if missing_cand_cols:
    raise KeyError(f"Missing columns in candidate_build_options_realistic.csv: {missing_cand_cols}")

print("Any zipcode missing req_total?", zip_df["req_total"].isna().sum())
print("Any zipcode missing req_under5?", zip_df["req_under5"].isna().sum())
print("Any facility with negative block cap?", (fac_df[["cap1", "cap2", "cap3"]] < 0).any().any())
print("Any candidate option blocked in this file?", int((cand_df.get("blocked_by_existing", 0) == 1).sum()))

Any zipcode missing req_total? 0
Any zipcode missing req_under5? 0
Any facility with negative block cap? False
Any candidate option blocked in this file? 0


In [8]:
# Build sets and parameter dictionaries
Z = zip_df["zipcode"].tolist()
F = fac_df["facility_id"].tolist()
candidate_sites = sorted(cand_df["candidate_id"].unique().tolist())
S = sorted(cand_df["size"].unique().tolist())

req_total = dict(zip(zip_df["zipcode"], zip_df["req_total"]))
req_under5 = dict(zip(zip_df["zipcode"], zip_df["req_under5"]))
existing_total = dict(zip(zip_df["zipcode"], zip_df["existing_total"]))
existing_under5 = dict(zip(zip_df["zipcode"], zip_df["existing_under5"]))

cap1 = dict(zip(fac_df["facility_id"], fac_df["cap1"]))
cap2 = dict(zip(fac_df["facility_id"], fac_df["cap2"]))
cap3 = dict(zip(fac_df["facility_id"], fac_df["cap3"]))
coef1 = dict(zip(fac_df["facility_id"], fac_df["coef1"]))
coef2 = dict(zip(fac_df["facility_id"], fac_df["coef2"]))
coef3 = dict(zip(fac_df["facility_id"], fac_df["coef3"]))

fac_by_zip = fac_df.groupby("zipcode")["facility_id"].apply(list).to_dict()
cand_key_tuples = list(zip(cand_df["candidate_id"], cand_df["size"]))
candsize_by_zip = cand_df.groupby("zipcode")[["candidate_id", "size"]].apply(lambda g: list(map(tuple, g.to_numpy()))).to_dict()
sizes_by_candidate = cand_df.groupby("candidate_id")["size"].apply(list).to_dict()

build_total = {(r["candidate_id"], r["size"]): r["new_total_capacity"] for _, r in cand_df.iterrows()}
build_under5_max = {(r["candidate_id"], r["size"]): r["new_under5_capacity_max"] for _, r in cand_df.iterrows()}
build_cost = {(r["candidate_id"], r["size"]): r["fixed_build_cost"] for _, r in cand_df.iterrows()}
blocked_candidates = set(ce_conflict_df["candidate_id"].unique().tolist()) if not ce_conflict_df.empty else set()

print("Number of zipcodes:", len(Z))
print("Number of existing facilities:", len(F))
print("Number of candidate sites:", len(candidate_sites))
print("Number of candidate-size options:", len(cand_key_tuples))
print("Blocked candidates:", len(blocked_candidates))

Number of zipcodes: 311
Number of existing facilities: 7712
Number of candidate sites: 28522
Number of candidate-size options: 85566
Blocked candidates: 2578


In [9]:
# Solver parameters and model setup
log_file = LOG_DIR / "realistic_model.log"

model = gp.Model("realistic_childcare")
model.Params.LogFile = str(log_file)
model.Params.MIPGap = 0.001
model.Params.TimeLimit = 600
model.Params.Presolve = 2
model.Params.Threads = 0
model.Params.OutputFlag = 1

Set parameter Username


Set parameter LicenseID to value 2773994


Academic license - for non-commercial use only - expires 2027-02-02


Set parameter LogFile to value "/Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/realistic/logs/realistic_model.log"


Set parameter MIPGap to value 0.001


Set parameter TimeLimit to value 600


Set parameter Presolve to value 2


Set parameter Threads to value 0


Set parameter OutputFlag to value 1


## Decision variables and constraints

The realistic model keeps facility-level expansion variables, but it replaces ZIP-code build counts with site-specific binary new-build decisions and explicit spacing constraints.


In [10]:
# Add decision variables
# Expansion variables for existing facilities by cost block
x1 = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="x1")
x2 = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="x2")
x3 = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="x3")
u = model.addVars(F, vtype=GRB.INTEGER, lb=0, name="u")

# New build variables by candidate site and size
y = model.addVars(cand_key_tuples, vtype=GRB.BINARY, name="y")
g = model.addVars(cand_key_tuples, vtype=GRB.INTEGER, lb=0, name="g")

In [11]:
# Facility-level expansion constraints
for f in F:
    model.addConstr(x1[f] <= cap1[f], name=f"cap1_{f}")
    model.addConstr(x2[f] <= cap2[f], name=f"cap2_{f}")
    model.addConstr(x3[f] <= cap3[f], name=f"cap3_{f}")
    model.addConstr(u[f] <= x1[f] + x2[f] + x3[f], name=f"under5_expand_cap_{f}")

In [12]:
# Candidate-site constraints
for l in candidate_sites:
    candidate_sizes = sizes_by_candidate.get(l, [])
    if candidate_sizes:
        model.addConstr(
            gp.quicksum(y[l, s] for s in candidate_sizes) <= 1,
            name=f"one_size_per_candidate_{l}",
        )

for (l, s) in cand_key_tuples:
    model.addConstr(
        g[l, s] <= build_under5_max[(l, s)] * y[l, s],
        name=f"new_under5_cap_{l}_{s}",
    )

for l in blocked_candidates:
    if l in sizes_by_candidate:
        model.addConstr(
            gp.quicksum(y[l, s] for s in sizes_by_candidate[l]) == 0,
            name=f"blocked_candidate_{l}",
        )

if not cc_conflict_df.empty:
    for _, row in cc_conflict_df.iterrows():
        l1 = int(row["candidate_id_1"])
        l2 = int(row["candidate_id_2"])

        sizes1 = sizes_by_candidate.get(l1, [])
        sizes2 = sizes_by_candidate.get(l2, [])

        if sizes1 and sizes2:
            model.addConstr(
                gp.quicksum(y[l1, s] for s in sizes1)
                + gp.quicksum(y[l2, s] for s in sizes2) <= 1,
                name=f"cc_conflict_{l1}_{l2}",
            )

In [13]:
# Zipcode-level service constraints
for z in Z:
    model.addConstr(
        existing_total[z]
        + gp.quicksum(x1[f] + x2[f] + x3[f] for f in fac_by_zip.get(z, []))
        + gp.quicksum(build_total[(l, s)] * y[l, s] for (l, s) in candsize_by_zip.get(z, []))
        >= req_total[z],
        name=f"total_req_{z}",
    )

for z in Z:
    model.addConstr(
        existing_under5[z]
        + gp.quicksum(u[f] for f in fac_by_zip.get(z, []))
        + gp.quicksum(g[l, s] for (l, s) in candsize_by_zip.get(z, []))
        >= req_under5[z],
        name=f"under5_req_{z}",
    )

## Objective function

The Part 2 objective combines fixed new-build costs, under-5 equipment costs, and additive block-specific expansion coefficients. This preserves increasing marginal expansion cost in a tractable MILP, even though it is an approximation to the assignment's literal piecewise-total rule.


In [14]:
# Set objective
SPECIAL_EQUIPMENT_COST = 100

# Canonical Part 2 notebook model: additive block costs with increasing marginal expansion coefficients
expansion_cost_expr = gp.quicksum(
    coef1[f] * x1[f] + coef2[f] * x2[f] + coef3[f] * x3[f]
    for f in F
)

new_build_cost_expr = gp.quicksum(
    build_cost[(l, s)] * y[l, s]
    for (l, s) in cand_key_tuples
)

under5_equipment_cost_expr = (
    gp.quicksum(SPECIAL_EQUIPMENT_COST * u[f] for f in F)
    + gp.quicksum(SPECIAL_EQUIPMENT_COST * g[l, s] for (l, s) in cand_key_tuples)
)

model.setObjective(
    expansion_cost_expr + new_build_cost_expr + under5_equipment_cost_expr,
    GRB.MINIMIZE,
)

In [15]:
# Model size check
model.update()

print("Number of variables:", model.NumVars)
print("Number of constraints:", model.NumConstrs)

Number of variables: 201980
Number of constraints: 155819


In [16]:
# Optimize
start_time = time.time()
model.optimize()
solve_time = time.time() - start_time

print(f"Solve time: {solve_time:.2f} seconds")
print("Model status code:", model.Status)

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D2128)


CPU model: Apple M4 Pro


Thread count: 12 physical cores, 12 logical processors, using up to 12 threads


Non-default parameters:


TimeLimit  600


...

Thread count was 12 (of 12 available processors)


Solution count 10: 1.83963e+08 1.84175e+08 1.84735e+08 ... 1.84903e+08


Optimal solution found (tolerance 1.00e-03)


Best objective 1.839632074983e+08, best bound 1.837983666003e+08, gap 0.0896%


Solve time: 18.21 seconds
Model status code: 2


In [17]:
# Interpret solve status and optional infeasibility diagnosis
status_map = {
    GRB.OPTIMAL: "OPTIMAL",
    GRB.TIME_LIMIT: "TIME_LIMIT",
    GRB.INFEASIBLE: "INFEASIBLE",
    GRB.INF_OR_UNBD: "INF_OR_UNBD",
    GRB.UNBOUNDED: "UNBOUNDED",
    GRB.SUBOPTIMAL: "SUBOPTIMAL",
    GRB.INTERRUPTED: "INTERRUPTED",
}

if model.Status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    model.computeIIS()
    iis_path = SOLUTIONS_DIR / "realistic_model.ilp"
    model.write(str(iis_path))
    print("IIS written to:", iis_path)

has_solution = model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT, GRB.SUBOPTIMAL] and model.SolCount > 0
print("Has solution:", has_solution)

if not has_solution:
    raise RuntimeError(f"No usable solution found. Status: {model.Status}")

Has solution: True


In [18]:
# Objective value and key cost components
objective_value = model.ObjVal
expansion_cost_value = expansion_cost_expr.getValue()
new_build_cost_value = new_build_cost_expr.getValue()
under5_equipment_cost_value = under5_equipment_cost_expr.getValue()

print("Objective value:", objective_value)
print("Expansion cost:", expansion_cost_value)
print("New build cost:", new_build_cost_value)
print("Under-5 equipment cost:", under5_equipment_cost_value)

Objective value: 183963207.49831694
Expansion cost: 9381107.498316955
New build cost: 146810000.0
Under-5 equipment cost: 27772100.0


In [19]:
# Facility expansion solution table
facility_solution = fac_df.copy()
facility_solution["x1_added"] = facility_solution["facility_id"].map(lambda f: x1[f].X)
facility_solution["x2_added"] = facility_solution["facility_id"].map(lambda f: x2[f].X)
facility_solution["x3_added"] = facility_solution["facility_id"].map(lambda f: x3[f].X)
facility_solution["u_under5_added"] = facility_solution["facility_id"].map(lambda f: u[f].X)
facility_solution["x_total_added"] = facility_solution["x1_added"] + facility_solution["x2_added"] + facility_solution["x3_added"]
facility_solution["selected_cost_band"] = np.select(
    [
        facility_solution["x3_added"] > 1e-6,
        facility_solution["x2_added"] > 1e-6,
        facility_solution["x1_added"] > 1e-6,
    ],
    ["15_20_pct", "10_15_pct", "0_10_pct"],
    default="no_expansion",
)
facility_solution["expansion_cost"] = (
    facility_solution["coef1"] * facility_solution["x1_added"]
    + facility_solution["coef2"] * facility_solution["x2_added"]
    + facility_solution["coef3"] * facility_solution["x3_added"]
)
facility_solution["under5_equipment_cost_expansion"] = 100 * facility_solution["u_under5_added"]
facility_solution["total_expansion_related_cost"] = (
    facility_solution["expansion_cost"] + facility_solution["under5_equipment_cost_expansion"]
)
facility_solution = facility_solution[facility_solution["x_total_added"] > 1e-6].copy()

print("Expanded facilities:", facility_solution.shape)
display(facility_solution.head())

Expanded facilities: (1126, 27)


,facility_id,facility_name,program_type,zipcode,latitude,longitude,n_f,b_f,cap1,cap2,cap3,cap10_cum,cap15_cum,cap20_cum,U_f_realistic,coef1,coef2,coef3,x1_added,x2_added,x3_added,u_under5_added,x_total_added,selected_cost_band,expansion_cost,under5_equipment_cost_expansion,total_expansion_related_cost
6,250057,KIPS BAY BOYS & GIRLS CLUB,SACC,10473,40.819564,-73.848827,240,0,24,12,12,24,36,48,48,283.333333,483.333333,1083.333333,24.0,-0.0,-0.0,24.0,24.0,0_10_pct,6800.000000,2400.0,9200.000000
19,624233,"Directions For Our Youth, Inc.",SACC,10462,40.844890,-73.858149,245,0,24,12,13,24,36,49,49,281.632653,481.632653,1081.632653,24.0,-0.0,-0.0,24.0,24.0,0_10_pct,6759.183673,2400.0,9159.183673
21,677764,Italian American Civil Rights League @MS 366,SACC,11236,40.647388,-73.891883,180,0,18,9,9,18,27,36,36,311.111111,511.111111,1111.111111,18.0,-0.0,-0.0,18.0,18.0,0_10_pct,5600.000000,1800.0,7400.000000
22,689575,"SCAN-Harbor, Inc.",SACC,10035,40.807100,-73.936369,40,0,4,2,2,4,6,8,8,700.000000,900.000000,1500.000000,4.0,2.0,0.0,6.0,6.0,10_15_pct,4600.000000,600.0,5200.000000
26,720363,"New York Edge, Inc.",SACC,10469,40.866987,-73.850800,206,0,20,10,11,20,30,41,41,297.087379,497.087379,1097.087379,20.0,10.0,11.0,41.0,41.0,15_20_pct,22980.582524,4100.0,27080.582524


In [20]:
# New build solution table and size mix
build_rows = []
for (l, s) in cand_key_tuples:
    if y[l, s].X > 0.5:
        row = cand_df[(cand_df["candidate_id"] == l) & (cand_df["size"] == s)].iloc[0].to_dict()
        row["build_selected"] = int(round(y[l, s].X))
        row["new_under5_assigned"] = g[l, s].X
        row["under5_equipment_cost_newbuild"] = 100 * g[l, s].X
        row["total_newbuild_related_cost"] = row["fixed_build_cost"] + row["under5_equipment_cost_newbuild"]
        build_rows.append(row)

new_build_solution = pd.DataFrame(build_rows)

if not new_build_solution.empty:
    build_size_summary = (
        new_build_solution.groupby("size", as_index=False)
        .agg(
            n_new_facilities=("candidate_id", "count"),
            total_capacity=("new_total_capacity", "sum"),
            total_under5=("new_under5_assigned", "sum"),
            fixed_build_cost=("fixed_build_cost", "sum"),
        )
        .sort_values("size")
    )
else:
    build_size_summary = pd.DataFrame(columns=["size", "n_new_facilities", "total_capacity", "total_under5", "fixed_build_cost"])

print("Selected new builds:", new_build_solution.shape)
display(new_build_solution.head())
display(build_size_summary)

Selected new builds: (1284, 14)


,candidate_id,zipcode,latitude,longitude,geo_missing_flag,size,new_total_capacity,new_under5_capacity_max,fixed_build_cost,blocked_by_existing,build_selected,new_under5_assigned,under5_equipment_cost_newbuild,total_newbuild_related_cost
0,43,10001,40.740596,-73.994574,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
1,85,10001,40.757723,-73.990862,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
2,101,10002,40.706000,-73.984906,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
3,103,10002,40.721387,-73.983519,False,large,400,200,115000,0,1,200.0,20000.0,135000.0
4,104,10002,40.724287,-73.990661,False,large,400,200,115000,0,1,200.0,20000.0,135000.0


,size,n_new_facilities,total_capacity,total_under5,fixed_build_cost
0,large,1261,504400,251111.0,145015000
1,medium,10,2000,955.0,950000
2,small,13,1300,588.0,845000


In [21]:
# Zipcode-level solution summary
if not facility_solution.empty:
    exp_zip = (
        facility_solution.groupby("zipcode", as_index=False)
        .agg(
            expanded_total=("x_total_added", "sum"),
            expanded_under5=("u_under5_added", "sum"),
            expansion_cost=("expansion_cost", "sum"),
            under5_equipment_cost_expansion=("under5_equipment_cost_expansion", "sum"),
            total_expansion_related_cost=("total_expansion_related_cost", "sum"),
        )
    )
else:
    exp_zip = pd.DataFrame(columns=[
        "zipcode", "expanded_total", "expanded_under5", "expansion_cost",
        "under5_equipment_cost_expansion", "total_expansion_related_cost"
    ])

if not new_build_solution.empty:
    new_build_zip = (
        new_build_solution.groupby("zipcode", as_index=False)
        .agg(
            n_new_facilities=("candidate_id", "count"),
            new_total_capacity=("new_total_capacity", "sum"),
            new_under5_capacity=("new_under5_assigned", "sum"),
            new_build_cost=("fixed_build_cost", "sum"),
            under5_equipment_cost_newbuild=("under5_equipment_cost_newbuild", "sum"),
            total_newbuild_related_cost=("total_newbuild_related_cost", "sum"),
        )
    )
else:
    new_build_zip = pd.DataFrame(columns=[
        "zipcode", "n_new_facilities", "new_total_capacity", "new_under5_capacity",
        "new_build_cost", "under5_equipment_cost_newbuild", "total_newbuild_related_cost"
    ])

zipcode_solution = zip_df.copy()
zipcode_solution = zipcode_solution.merge(exp_zip, on="zipcode", how="left")
zipcode_solution = zipcode_solution.merge(new_build_zip, on="zipcode", how="left")

fill_zero_cols = [
    "expanded_total", "expanded_under5", "expansion_cost", "under5_equipment_cost_expansion",
    "total_expansion_related_cost", "n_new_facilities", "new_total_capacity", "new_under5_capacity",
    "new_build_cost", "under5_equipment_cost_newbuild", "total_newbuild_related_cost",
]
for c in fill_zero_cols:
    zipcode_solution[c] = zipcode_solution[c].fillna(0)

zipcode_solution["final_total_capacity"] = (
    zipcode_solution["existing_total"]
    + zipcode_solution["expanded_total"]
    + zipcode_solution["new_total_capacity"]
)
zipcode_solution["final_under5_capacity"] = (
    zipcode_solution["existing_under5"]
    + zipcode_solution["expanded_under5"]
    + zipcode_solution["new_under5_capacity"]
)
zipcode_solution["total_slack"] = zipcode_solution["final_total_capacity"] - zipcode_solution["req_total"]
zipcode_solution["under5_slack"] = zipcode_solution["final_under5_capacity"] - zipcode_solution["req_under5"]
zipcode_solution["all_requirements_met_after"] = (
    (zipcode_solution["total_slack"] >= -1e-6)
    & (zipcode_solution["under5_slack"] >= -1e-6)
).astype(int)

display(zipcode_solution.head())

,zipcode,pop_0_5,pop_6_12,child_pop_0_12,employment_rate,average_income,employment_missing_flag,income_missing_flag,pop_0_5_missing_flag,pop_6_12_missing_flag,child_pop_0_12_missing_flag,high_demand,total_threshold_rule,req_total,req_under5,existing_total,existing_under5,n_active_facilities,gap_total,gap_under5,is_desert_total_before,is_under5_short_before,needs_intervention_before,expanded_total,expanded_under5,expansion_cost,under5_equipment_cost_expansion,total_expansion_related_cost,n_new_facilities,new_total_capacity,new_under5_capacity,new_build_cost,under5_equipment_cost_newbuild,total_newbuild_related_cost,final_total_capacity,final_under5_capacity,total_slack,under5_slack,all_requirements_met_after
0,10001,744.0,1255.0,1999.0,0.595097,102878.033603,0,0,0,0,0,0,666.333333,667,496,609.0,24.0,9.0,58.0,472.0,1,1,1,72.0,72.0,28214.759976,7200.0,35414.759976,2.0,800.0,400.0,230000.0,40000.0,270000.0,1481.0,496.0,814.0,0.0,1
1,10002,2142.0,4645.0,6787.0,0.520662,59604.041165,0,0,0,0,0,1,3393.500000,3394,1428,4724.0,198.0,56.0,0.0,1230.0,0,1,1,630.0,630.0,244439.557306,63000.0,307439.557306,3.0,1200.0,600.0,345000.0,60000.0,405000.0,6554.0,1428.0,3160.0,0.0,1
2,10003,1440.0,1510.0,2950.0,0.497244,114273.049645,0,0,0,0,0,0,983.333333,984,960,1995.0,0.0,7.0,0.0,960.0,0,1,1,160.0,160.0,39762.682751,16000.0,55762.682751,4.0,1600.0,800.0,460000.0,80000.0,540000.0,3755.0,960.0,2771.0,0.0,1
3,10004,433.0,262.0,695.0,0.506661,132004.310345,0,0,0,0,0,0,231.666667,232,289,263.0,0.0,2.0,0.0,289.0,0,1,1,39.0,39.0,14920.000000,3900.0,18820.000000,2.0,500.0,250.0,180000.0,25000.0,205000.0,802.0,289.0,570.0,0.0,1
4,10005,484.0,318.0,802.0,0.665833,121437.713311,0,0,0,0,0,1,401.000000,402,323,39.0,0.0,1.0,363.0,323.0,1,1,1,0.0,0.0,0.000000,0.0,0.000000,2.0,800.0,323.0,230000.0,32300.0,262300.0,839.0,323.0,437.0,0.0,1


In [22]:
# Objective breakdown table
objective_breakdown = pd.DataFrame([
    {"component": "expansion_cost", "value": expansion_cost_value},
    {"component": "new_build_cost", "value": new_build_cost_value},
    {"component": "under5_equipment_cost", "value": under5_equipment_cost_value},
    {"component": "total_objective", "value": objective_value},
])

display(objective_breakdown)

,component,value
0,expansion_cost,9.381107e+06
1,new_build_cost,1.468100e+08
2,under5_equipment_cost,2.777210e+07
3,total_objective,1.839632e+08


In [23]:
# Run metadata and summary statistics
size_counts = new_build_solution["size"].value_counts().to_dict() if not new_build_solution.empty else {}

run_metadata = pd.DataFrame([
    {"metric": "model_status_code", "value": model.Status},
    {"metric": "model_status_text", "value": status_map.get(model.Status, str(model.Status))},
    {"metric": "objective_value", "value": model.ObjVal},
    {"metric": "best_bound", "value": model.ObjBound},
    {"metric": "mip_gap", "value": model.MIPGap if model.IsMIP else 0},
    {"metric": "solve_seconds", "value": solve_time},
    {"metric": "num_variables", "value": model.NumVars},
    {"metric": "num_constraints", "value": model.NumConstrs},
    {"metric": "solution_count", "value": model.SolCount},
    {"metric": "cost_mode", "value": "additive_blocks"},
    {"metric": "time_limit_seconds", "value": model.Params.TimeLimit},
])

summary_stats = pd.DataFrame([
    {"metric": "n_expanded_facilities", "value": len(facility_solution)},
    {"metric": "total_expansion_slots", "value": facility_solution["x_total_added"].sum() if not facility_solution.empty else 0},
    {"metric": "total_expansion_under5_slots", "value": facility_solution["u_under5_added"].sum() if not facility_solution.empty else 0},
    {"metric": "n_new_facilities", "value": len(new_build_solution) if not new_build_solution.empty else 0},
    {"metric": "total_new_capacity", "value": new_build_solution["new_total_capacity"].sum() if not new_build_solution.empty else 0},
    {"metric": "total_new_under5_capacity", "value": new_build_solution["new_under5_assigned"].sum() if not new_build_solution.empty else 0},
    {"metric": "n_new_small_facilities", "value": size_counts.get("small", 0)},
    {"metric": "n_new_medium_facilities", "value": size_counts.get("medium", 0)},
    {"metric": "n_new_large_facilities", "value": size_counts.get("large", 0)},
    {"metric": "zipcodes_with_new_facilities", "value": int((zipcode_solution["n_new_facilities"] > 0).sum())},
    {"metric": "zipcodes_with_expansion", "value": int((zipcode_solution["expanded_total"] > 0).sum())},
    {"metric": "zipcodes_zero_total_slack", "value": int((zipcode_solution["total_slack"].abs() < 1e-6).sum())},
    {"metric": "zipcodes_zero_under5_slack", "value": int((zipcode_solution["under5_slack"].abs() < 1e-6).sum())},
    {"metric": "zipcodes_meeting_all_requirements_after", "value": zipcode_solution["all_requirements_met_after"].sum()},
])

display(run_metadata)
display(summary_stats)

,metric,value
0,model_status_code,2
1,model_status_text,OPTIMAL
2,objective_value,183963207.498317
3,best_bound,183798366.600326
4,mip_gap,0.000896
5,solve_seconds,18.213744
6,num_variables,201980
7,num_constraints,155819
8,solution_count,10
9,cost_mode,additive_blocks


,metric,value
0,n_expanded_facilities,1126.0
1,total_expansion_slots,25108.0
2,total_expansion_under5_slots,25067.0
3,n_new_facilities,1284.0
4,total_new_capacity,507700.0
5,total_new_under5_capacity,252654.0
6,n_new_small_facilities,13.0
7,n_new_medium_facilities,10.0
8,n_new_large_facilities,1261.0
9,zipcodes_with_new_facilities,180.0


## Post-solve validation checks

These checks verify that the exported Part 2 solution satisfies the coded service and siting rules, rather than merely returning a numerically feasible-looking plan.


In [24]:
# Post-solve validation checks
selected_candidates = set(pd.to_numeric(new_build_solution["candidate_id"], errors="coerce").dropna().astype(int)) if not new_build_solution.empty else set()
blocked_candidates = set(pd.to_numeric(ce_conflict_df["candidate_id"], errors="coerce").dropna().astype(int)) if not ce_conflict_df.empty else set()
blocked_selected_candidates = sorted(selected_candidates & blocked_candidates)

candidate_conflict_violations = pd.DataFrame(columns=cc_conflict_df.columns)
if not cc_conflict_df.empty and selected_candidates:
    candidate_conflict_violations = cc_conflict_df[
        cc_conflict_df["candidate_id_1"].isin(selected_candidates)
        & cc_conflict_df["candidate_id_2"].isin(selected_candidates)
    ].copy()

facility_validation = fac_df[["facility_id", "cap1", "cap2", "cap3"]].merge(
    facility_solution[["facility_id", "x1_added", "x2_added", "x3_added", "x_total_added", "u_under5_added"]],
    on="facility_id",
    how="left",
)
for col in ["x1_added", "x2_added", "x3_added", "x_total_added", "u_under5_added"]:
    facility_validation[col] = facility_validation[col].fillna(0.0)

facility_cap_violations = facility_validation[
    (facility_validation["x1_added"] - facility_validation["cap1"] > 1e-6)
    | (facility_validation["x2_added"] - facility_validation["cap2"] > 1e-6)
    | (facility_validation["x3_added"] - facility_validation["cap3"] > 1e-6)
    | (facility_validation["u_under5_added"] - facility_validation["x_total_added"] > 1e-6)
].copy()

zipcode_requirement_violations = zipcode_solution[
    (zipcode_solution["total_slack"] < -1e-6)
    | (zipcode_solution["under5_slack"] < -1e-6)
    | (zipcode_solution["all_requirements_met_after"] != 1)
].copy()

zero_child_zipcodes_with_new_builds = zipcode_solution[
    (zipcode_solution["child_pop_0_12"].fillna(0) <= 0)
    & (zipcode_solution["n_new_facilities"] > 0)
].copy()

validation_checks = pd.DataFrame([
    {"check": "zipcodes_meeting_service_requirements", "value": int((zipcode_solution["all_requirements_met_after"] == 1).sum()), "target": len(zipcode_solution), "violations": len(zipcode_requirement_violations), "passed": len(zipcode_requirement_violations) == 0},
    {"check": "blocked_candidates_selected", "value": len(blocked_selected_candidates), "target": 0, "violations": len(blocked_selected_candidates), "passed": len(blocked_selected_candidates) == 0},
    {"check": "candidate_conflict_pair_violations", "value": len(candidate_conflict_violations), "target": 0, "violations": len(candidate_conflict_violations), "passed": len(candidate_conflict_violations) == 0},
    {"check": "facility_cap_or_under5_violations", "value": len(facility_cap_violations), "target": 0, "violations": len(facility_cap_violations), "passed": len(facility_cap_violations) == 0},
    {"check": "zero_child_zipcodes_with_new_builds", "value": len(zero_child_zipcodes_with_new_builds), "target": 0, "violations": len(zero_child_zipcodes_with_new_builds), "passed": len(zero_child_zipcodes_with_new_builds) == 0},
])
validation_checks["passed"] = validation_checks["passed"].astype(int)

print("All validation checks passed:", bool((validation_checks["passed"] == 1).all()))
display(validation_checks)


All validation checks passed: True


,check,value,target,violations,passed
0,zipcodes_meeting_service_requirements,311,311,0,1
1,blocked_candidates_selected,0,0,0,1
2,candidate_conflict_pair_violations,0,0,0,1
3,facility_cap_or_under5_violations,0,0,0,1
4,zero_child_zipcodes_with_new_builds,0,0,0,1


In [25]:
# Quick management summary for the report
print("Top 15 zipcodes by new total capacity:")
display(
    zipcode_solution.sort_values("new_total_capacity", ascending=False)[[
        "zipcode", "new_total_capacity", "expanded_total",
        "req_total", "req_under5", "total_slack", "under5_slack"
    ]].head(15)
)

print("Top 15 facilities by added expansion:")
display(
    facility_solution.sort_values("x_total_added", ascending=False)[[
        "facility_id", "facility_name", "zipcode",
        "x1_added", "x2_added", "x3_added",
        "x_total_added", "u_under5_added", "expansion_cost"
    ]].head(15)
)

print("Build size mix:")
display(build_size_summary)

Top 15 zipcodes by new total capacity:


,zipcode,new_total_capacity,expanded_total,req_total,req_under5,total_slack,under5_slack
213,11219,14000.0,72.0,13374,8084,3293.0,0.0
223,11230,10000.0,0.0,5803,5279,5021.0,0.0
258,11368,8800.0,361.0,9621,5263,2597.0,0.0
200,11206,8400.0,244.0,8618,4802,3259.0,0.0
217,11223,8400.0,84.0,7741,4428,2191.0,0.0
198,11204,8000.0,374.0,8683,4806,2883.0,0.0
202,11208,7600.0,500.0,9814,5518,3340.0,0.0
159,10314,6400.0,157.0,4624,3513,3587.0,0.0
214,11220,6400.0,259.0,8344,3855,566.0,0.0
271,11385,6400.0,208.0,7290,4018,2292.0,0.0


Top 15 facilities by added expansion:


,facility_id,facility_name,zipcode,x1_added,x2_added,x3_added,x_total_added,u_under5_added,expansion_cost
3057,481959,"New York Center for Interpersonal Development,...",10314,89.0,41.0,-0.0,130.0,130.0,37121.348315
1257,121475,LATCH AFTERSCHOOL PROGRAM AT P.S. 89Q,11373,72.0,36.0,-0.0,108.0,108.0,31800.000000
3806,839931,"Manhattan Youth Recreation & Resources, Inc.",10014,58.0,29.0,17.0,104.0,104.0,43755.555556
1665,73952,CYPRESS HILLS LOCAL DEVELOPEMENT CORP.,11208,67.0,33.0,-0.0,100.0,100.0,29580.625931
2654,479327,SoBRO,10460,50.0,25.0,22.0,97.0,97.0,45880.000000
6641,901628,Yeshivath Kehilath Yakov,11218,49.0,24.0,18.0,91.0,91.0,41106.720978
4836,73515,The Children's Aid Society @I.S. 218,10040,47.0,23.0,20.0,90.0,90.0,42413.559322
6625,920666,"Imogen Roche Foundation, Inc.",10014,57.0,29.0,0.0,86.0,86.0,25996.515679
1118,443292,"Casita Maria, Inc.",10459,56.0,28.0,0.0,84.0,84.0,25394.652406
2488,74233,SAMUEL FIELD Y/BAY TERRACE BEACON JHS 216,11365,44.0,22.0,18.0,84.0,84.0,39418.181818


Build size mix:


,size,n_new_facilities,total_capacity,total_under5,fixed_build_cost
0,large,1261,504400,251111.0,145015000
1,medium,10,2000,955.0,950000
2,small,13,1300,588.0,845000


In [26]:
# Export all realistic solution files
facility_solution.to_csv(SOLUTIONS_DIR / "facility_solution_realistic.csv", index=False)
zipcode_solution.to_csv(SOLUTIONS_DIR / "zipcode_solution_realistic.csv", index=False)
objective_breakdown.to_csv(SOLUTIONS_DIR / "objective_breakdown_realistic.csv", index=False)
run_metadata.to_csv(SOLUTIONS_DIR / "run_metadata_realistic.csv", index=False)
summary_stats.to_csv(SOLUTIONS_DIR / "summary_stats_realistic.csv", index=False)
build_size_summary.to_csv(SOLUTIONS_DIR / "build_size_summary_realistic.csv", index=False)
validation_checks.to_csv(SOLUTIONS_DIR / "validation_checks_realistic.csv", index=False)

if len(new_build_solution) > 0:
    new_build_solution.to_csv(SOLUTIONS_DIR / "new_build_solution_realistic.csv", index=False)
else:
    pd.DataFrame(columns=cand_df.columns.tolist() + [
        "build_selected",
        "new_under5_assigned",
        "under5_equipment_cost_newbuild",
        "total_newbuild_related_cost",
    ]).to_csv(SOLUTIONS_DIR / "new_build_solution_realistic.csv", index=False)

print("Realistic solution files saved to:", SOLUTIONS_DIR)

Realistic solution files saved to: /Users/alexandresepulvedadedietrich/Documents/School/Columbia/Spring_2026/Optimization/group_project/results/realistic/solutions
